# 07 · Figures 1–4

Drawn with the package's own figure functions (`vmlevaluator.figures.paper_figures`), so the
theme matches the rest of the paper; `scripts/manuscript_figures.py` overrides the model list, the
panel titles, the bar labels (three decimals, as in the text), the "Fusion" legends and the
shared ROC legend for this process only.

With `RUN = False` the bars are drawn from the manuscript's frozen numbers and the ROC
curves from the manuscript run's predictions; the result is pixel-identical to the
figures in the manuscript (checked in the last cell).

This notebook needs no images: it draws from the saved predictions in `models/`.
`RUN = True` draws from a re-run of the original data in `runs/rerun` instead; notebook 09
draws the figures for a synthetic run.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
SYNTHETIC = False
os.environ.pop("VML_SYNTHETIC", None)   # always the manuscript's saved predictions
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
from manuscript_figures import make_figures
from manuscript_numbers import compute
N = compute(BASE) if RUN else None            # None -> reference/manuscript_numbers.json
OUT = P.WORK / ("figures_rerun" if RUN else "figures")
paths = make_figures(N, OUT, BASE)
paths

In [ ]:
from IPython.display import Image, display
for name in ["figure1_patch_f1", "figure2_roc", "figure3_wsi_f1", "figure4_aclr_transfer"]:
    print(name); display(Image(filename=str(OUT / f"{name}.png"), width=900))

In [ ]:
# Compare with the figures in the manuscript (figures/manuscript/).
from PIL import Image as PImage, ImageChops
ms = P.MANUSCRIPT_FIGURES
for name in ["figure1_patch_f1", "figure2_roc", "figure3_wsi_f1", "figure4_aclr_transfer"]:
    if not (ms / f"{name}.png").is_file():
        print(f"{name}: manuscript copy not found at {ms}"); continue
    a, b = PImage.open(ms / f"{name}.png").convert("RGB"), PImage.open(OUT / f"{name}.png").convert("RGB")
    same = a.size == b.size and ImageChops.difference(a, b).getbbox() is None
    print(f"{name:24s} {'pixel-identical to the manuscript figure' if same else 'differs from the manuscript figure'}")